## 1. Locate a zipped or automatically unzipped input package


In [ ]:
from pathlib import Path
import base64, hashlib, json, os, shutil, subprocess, sys, time, zipfile
INPUT_ROOT = Path(os.environ.get("JCAM_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("JCAM_WORK_ROOT", "/kaggle/working"))
RUN_ROOT = WORK_ROOT / "jcam_thm06"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
STAGES = []
RUN_OK = True
PACKAGE_MODE = "embedded_notebook_sources"
def record(stage, outcome, detail):
    STAGES.append({"stage": stage, "outcome": outcome, "detail": detail})
    print(stage, outcome, detail)
try:
    candidates = sorted(INPUT_ROOT.rglob("pareto_information_lower_bound.py")) if INPUT_ROOT.exists() else []
    if candidates:
        PACKAGE_MODE = "kaggle_auto_unzipped"
    elif INPUT_ROOT.exists():
        for path in sorted(INPUT_ROOT.rglob("*.zip")):
            try:
                with zipfile.ZipFile(path) as archive:
                    members = [name for name in archive.namelist() if name.replace("\\", "/").endswith("pareto_information_lower_bound.py")]
                    if not members:
                        continue
                    destination = RUN_ROOT / "input_package"
                    destination.mkdir(parents=True, exist_ok=True)
                    for member in archive.infolist():
                        target = (destination / member.filename).resolve()
                        if not target.is_relative_to(destination.resolve()):
                            raise RuntimeError(f"Unsafe ZIP member: {member.filename}")
                    archive.extractall(destination)
                    PACKAGE_MODE = "kaggle_zip_extracted"
                    break
            except zipfile.BadZipFile:
                continue
    record("bootstrap", "passed", PACKAGE_MODE)
except Exception as error:
    RUN_OK = False
    record("bootstrap", "failed", repr(error))

## 2. Install the exact experiment and proof sources from notebook cells


In [ ]:
EXPERIMENT_SOURCE = base64.b64decode("".join([
    'IiIiQ29udHJvbGxlZCBudW1lcmljYWwgY2hlY2sgb2YgVEhNLTA2LzA3J3Mgc2V0LWNvdmVyYWdlIGluZm9ybWF0aW9uIHBy',
    'aWNlLgoKVGhpcyBjaGVja3MgdGhlIHRoZW9yZW0ncyBleGFjdCBxdWFkcmF0aWMgZmFtaWx5IGFnYWluc3QgYSBmaW5pdGUg',
    'Z3JpZCBvZgpoaWRkZW4gd29ybGRzIGFuZCBhIGRlbnNlIGdyaWQgb3ZlciBlYWNoIHRydWUgUGFyZXRvIGludGVydmFsLiBU',
    'aGUgZ3JpZAptYXhpbXVtIGlzIGEgbG93ZXIgYXBwcm94aW1hdGlvbiB0byBjb250aW51b3VzIGNvdmVyYWdlOyBpdCBpcyBu',
    'b3QgYSBwcm9vZi4KIiIiCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGNzdgppbXBvcnQganNvbgpmcm9tIHBhdGhsaWIgaW1w',
    'b3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAoKClJBRElVUyA9IDEuMCAvIDIwLjAKCgpkZWYgc2FtcGxlZF9jb3ZlcmFn',
    'ZSh0aGV0YTogbnAubmRhcnJheSwgYWN0aW9uczogbnAubmRhcnJheSwgYW1wbGl0dWRlOiBmbG9hdCwKICAgICAgICAgICAg',
    'ICAgICAgICAgdV9jb3VudDogaW50ID0gMTAxKSAtPiBucC5uZGFycmF5OgogICAgIiIiUmV0dXJuIGdyaWQgYXBwcm94aW1h',
    'dGlvbiB0byBUSE0tMDUgY292ZXJhZ2UgZm9yIGVhY2ggaGlkZGVuIHdvcmxkLiIiIgogICAgb2Zmc2V0cyA9IG5wLmxpbnNw',
    'YWNlKC1SQURJVVMsIFJBRElVUywgdV9jb3VudCkKICAgIHRhcmdldCA9IHRoZXRhWzosIE5vbmUsIE5vbmVdICsgb2Zmc2V0',
    'c1tOb25lLCA6LCBOb25lXQogICAgY2VudGVycyA9IG5wLnN0YWNrKCh0aGV0YSAtIFJBRElVUywgdGhldGEgKyBSQURJVVMp',
    'LCBheGlzPS0xKQogICAgIyBhY3Rpb25zIGhhcyBvbmUgcm93IHBlciB3b3JsZCBhbmQgYSBjb21tb24gc2V0IGJ1ZGdldCBL',
    'LgogICAgYWN0aW9uID0gYWN0aW9uc1s6LCBOb25lLCA6LCBOb25lXQogICAgdGFyZ2V0ID0gdGFyZ2V0WzosIDosIDosIE5v',
    'bmVdCiAgICBjZW50ZXJzID0gY2VudGVyc1s6LCBOb25lLCBOb25lLCA6XQogICAgZGVmaWNpdCA9IGFtcGxpdHVkZSAqICgo',
    'YWN0aW9uIC0gY2VudGVycykgKiogMiAtICh0YXJnZXQgLSBjZW50ZXJzKSAqKiAyKQogICAgcmV0dXJuIG5wLm1heGltdW0o',
    'ZGVmaWNpdCwgMC4wKS5tYXgoYXhpcz0tMSkubWluKGF4aXM9LTEpLm1heChheGlzPS0xKQoKCmRlZiBydW4ob3V0cHV0OiBQ',
    'YXRoLCB3b3JsZF9jb3VudDogaW50ID0gMTAyNCwgdV9jb3VudDogaW50ID0gMTAxLAogICAgICAgIGhvcml6b246IGludCA9',
    'IDEyOCkgLT4gZGljdDoKICAgIGlmIHdvcmxkX2NvdW50IDwgMTYgb3Igd29ybGRfY291bnQgJSAxNiBvciB1X2NvdW50IDwg',
    'MyBvciB1X2NvdW50ICUgMiA9PSAwOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIndvcmxkX2NvdW50IG11c3QgYmUgYSBt',
    'dWx0aXBsZSBvZiAxNjsgdV9jb3VudCBtdXN0IGJlIG9kZCIpCiAgICBpZiBob3Jpem9uIDwgMToKICAgICAgICByYWlzZSBW',
    'YWx1ZUVycm9yKCJob3Jpem9uIG11c3QgYmUgcG9zaXRpdmUiKQogICAgbGVuZ3RoID0gMS4wIC0gMi4wICogUkFESVVTCiAg',
    'ICB0aGV0YSA9IFJBRElVUyArIGxlbmd0aCAqIChucC5hcmFuZ2Uod29ybGRfY291bnQpICsgMC41KSAvIHdvcmxkX2NvdW50',
    'CiAgICByb3dzID0gW10KICAgIGZvciBhbXBsaXR1ZGUgaW4gKDAuMTI1LCAwLjI1LCAwLjUsIDEuMCwgMi4wKToKICAgICAg',
    'ICBmb3IgYnVkZ2V0IGluICgxLCAyLCA0LCA4LCAxNik6CiAgICAgICAgICAgIGJsaW5kID0gUkFESVVTICsgbGVuZ3RoICog',
    'KG5wLmFyYW5nZShidWRnZXQpICsgMC41KSAvIGJ1ZGdldAogICAgICAgICAgICBibGluZF9zZXRzID0gbnAuYnJvYWRjYXN0',
    'X3RvKGJsaW5kLCAod29ybGRfY291bnQsIGJ1ZGdldCkpCiAgICAgICAgICAgIHVuaXZlcnNhbCA9IChucC5hcmFuZ2UoYnVk',
    'Z2V0KSArIDAuNSkgLyBidWRnZXQKICAgICAgICAgICAgdW5pdmVyc2FsX3NldHMgPSBucC5icm9hZGNhc3RfdG8odW5pdmVy',
    'c2FsLCAod29ybGRfY291bnQsIGJ1ZGdldCkpCiAgICAgICAgICAgIG9yYWNsZV9zZXRzID0gdGhldGFbOiwgTm9uZV0gKyBS',
    'QURJVVMgKiAoMiAqIChucC5hcmFuZ2UoYnVkZ2V0KSArIDAuNSkgLyBidWRnZXQgLSAxKQogICAgICAgICAgICBibGluZF9s',
    'b3NzID0gc2FtcGxlZF9jb3ZlcmFnZSh0aGV0YSwgYmxpbmRfc2V0cywgYW1wbGl0dWRlLCB1X2NvdW50KQogICAgICAgICAg',
    'ICB1bml2ZXJzYWxfbG9zcyA9IHNhbXBsZWRfY292ZXJhZ2UodGhldGEsIHVuaXZlcnNhbF9zZXRzLCBhbXBsaXR1ZGUsIHVf',
    'Y291bnQpCiAgICAgICAgICAgIG9yYWNsZV9sb3NzID0gc2FtcGxlZF9jb3ZlcmFnZSh0aGV0YSwgb3JhY2xlX3NldHMsIGFt',
    'cGxpdHVkZSwgdV9jb3VudCkKICAgICAgICAgICAgZXhjZXNzID0gYmxpbmRfbG9zcyAtIG9yYWNsZV9sb3NzCiAgICAgICAg',
    'ICAgIGd1YXJhbnRlZWQgPSBhbXBsaXR1ZGUgLyAoODAgKiBidWRnZXQpCiAgICAgICAgICAgIHVwcGVyID0gYW1wbGl0dWRl',
    'IC8gYnVkZ2V0CiAgICAgICAgICAgICMgVGhlIG9yYWNsZSBncmlkIGlzIGZlYXNpYmxlIGJ1dCBuZWVkIG5vdCBiZSB0aGUg',
    'dHJ1ZSBiZXN0IEstc2V0LgogICAgICAgICAgICAjIFJlcGxhY2luZyBpdCBieSB0aGUgdHJ1ZSBvcHRpbXVtIG9ubHkgaW5j',
    'cmVhc2VzIHRoZSBleGNlc3MuCiAgICAgICAgICAgIHJvd3MuYXBwZW5kKHsKICAgICAgICAgICAgICAgICJhbXBsaXR1ZGUi',
    'OiBhbXBsaXR1ZGUsICJLIjogYnVkZ2V0LCAiVCI6IGhvcml6b24sCiAgICAgICAgICAgICAgICAid29ybGRfY291bnQiOiB3',
    'b3JsZF9jb3VudCwgInVfY291bnQiOiB1X2NvdW50LAogICAgICAgICAgICAgICAgIm1lYW5fYmxpbmRfY292ZXJhZ2UiOiBm',
    'bG9hdChibGluZF9sb3NzLm1lYW4oKSksCiAgICAgICAgICAgICAgICAibWVhbl91bml2ZXJzYWxfZ3JpZF9jb3ZlcmFnZSI6',
    'IGZsb2F0KHVuaXZlcnNhbF9sb3NzLm1lYW4oKSksCiAgICAgICAgICAgICAgICAibWF4X3VuaXZlcnNhbF9ncmlkX2NvdmVy',
    'YWdlIjogZmxvYXQodW5pdmVyc2FsX2xvc3MubWF4KCkpLAogICAgICAgICAgICAgICAgInRoZW9yZW1fdW5pZm9ybV91cHBl',
    'cl9ib3VuZCI6IHVwcGVyLAogICAgICAgICAgICAgICAgInVwcGVyX251bWVyaWNhbF9jaGVja19ob2xkcyI6IGJvb2wodW5p',
    'dmVyc2FsX2xvc3MubWF4KCkgPD0gdXBwZXIgKyAxZS0xMiksCiAgICAgICAgICAgICAgICAib3JhY2xlX2dyaWRfY292ZXJh',
    'Z2UiOiBmbG9hdChvcmFjbGVfbG9zcy5tZWFuKCkpLAogICAgICAgICAgICAgICAgIm1lYW5fZXhjZXNzX292ZXJfb3JhY2xl',
    'X2dyaWQiOiBmbG9hdChleGNlc3MubWVhbigpKSwKICAgICAgICAgICAgICAgICJ0aGVvcmVtX2V4Y2Vzc19sb3dlcl9ib3Vu',
    'ZCI6IGd1YXJhbnRlZWQsCiAgICAgICAgICAgICAgICAibnVtZXJpY2FsX2NoZWNrX2hvbGRzIjogYm9vbChleGNlc3MubWVh',
    'bigpICsgMWUtMTIgPj0gZ3VhcmFudGVlZCksCiAgICAgICAgICAgICAgICAibWVhbl9jdW11bGF0aXZlX2V4Y2VzcyI6IGZs',
    'b2F0KGhvcml6b24gKiBleGNlc3MubWVhbigpKSwKICAgICAgICAgICAgICAgICJ0aGVvcmVtX2N1bXVsYXRpdmVfbG93ZXJf',
    'Ym91bmQiOiBob3Jpem9uICogZ3VhcmFudGVlZCwKICAgICAgICAgICAgfSkKICAgIG91dHB1dC5ta2RpcihwYXJlbnRzPVRy',
    'dWUsIGV4aXN0X29rPVRydWUpCiAgICB3aXRoIChvdXRwdXQgLyAicGFyZXRvX2luZm9ybWF0aW9uX2xvd2VyX2JvdW5kLmNz',
    'diIpLm9wZW4oInciLCBuZXdsaW5lPSIiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBoYW5kbGU6CiAgICAgICAgd3JpdGVyID0g',
    'Y3N2LkRpY3RXcml0ZXIoaGFuZGxlLCBmaWVsZG5hbWVzPWxpc3Qocm93c1swXSkpCiAgICAgICAgd3JpdGVyLndyaXRlaGVh',
    'ZGVyKCkKICAgICAgICB3cml0ZXIud3JpdGVyb3dzKHJvd3MpCiAgICBzdW1tYXJ5ID0gewogICAgICAgICJ0aGVvcmVtIjog',
    'IlRITS0wNiBhbmQgVEhNLTA3IiwKICAgICAgICAicHJvdG9jb2wiOiAiaWRlbnRpY2FsIGNvbXBsZXRlIGludGVydmFscyBb',
    'MCxhXTsgaGlkZGVuIGZpeGVkIHRyYW5zbGF0ZWQgUGFyZXRvIGludGVydmFsIiwKICAgICAgICAibWV0cmljIjogIlRITS0w',
    'NSBzdXAtaW5mLXBvc2l0aXZlLXBhcnQgc2V0IGNvdmVyYWdlIiwKICAgICAgICAidGhlb3JldGljYWxfc3RhdGVtZW50Ijog',
    'IlVfVC8oODBLKSA8PSBtaW5pbWF4IGV4cGVjdGVkIGV4Y2VzcyBvdmVyIG9wdGltYWwgd29ybGQtc3BlY2lmaWMgSy1zZXQg',
    'PD0gVV9UL0siLAogICAgICAgICJudW1lcmljYWxfbWV0aG9kIjogIm1pZHBvaW50IHdvcmxkIHF1YWRyYXR1cmUgYW5kIGRl',
    'bnNlIFBhcmV0by10YXJnZXQgZ3JpZCIsCiAgICAgICAgIm51bWVyaWNhbF9yZXN1bHRzX2FyZV9wcm9vZiI6IEZhbHNlLAog',
    'ICAgICAgICJjb25maWd1cmF0aW9ucyI6IGxlbihyb3dzKSwKICAgICAgICAiYWxsX2NoZWNrZWRfY29uZmlndXJhdGlvbnNf',
    'bWVldF9ib3VuZCI6IGFsbChyb3dbIm51bWVyaWNhbF9jaGVja19ob2xkcyJdIGZvciByb3cgaW4gcm93cyksCiAgICAgICAg',
    'ImFsbF9jaGVja2VkX2NvbmZpZ3VyYXRpb25zX21lZXRfdXBwZXJfYm91bmQiOiBhbGwocm93WyJ1cHBlcl9udW1lcmljYWxf',
    'Y2hlY2tfaG9sZHMiXSBmb3Igcm93IGluIHJvd3MpLAogICAgICAgICJzbWFsbGVzdF9leGNlc3NfdG9fYm91bmRfcmF0aW8i',
    'OiBtaW4oCiAgICAgICAgICAgIHJvd1sibWVhbl9leGNlc3Nfb3Zlcl9vcmFjbGVfZ3JpZCJdIC8gcm93WyJ0aGVvcmVtX2V4',
    'Y2Vzc19sb3dlcl9ib3VuZCJdIGZvciByb3cgaW4gcm93cyksCiAgICAgICAgIm1lYW5fZXhjZXNzX3RvX2JvdW5kX3JhdGlv',
    'IjogZmxvYXQobnAubWVhbihbCiAgICAgICAgICAgIHJvd1sibWVhbl9leGNlc3Nfb3Zlcl9vcmFjbGVfZ3JpZCJdIC8gcm93',
    'WyJ0aGVvcmVtX2V4Y2Vzc19sb3dlcl9ib3VuZCJdIGZvciByb3cgaW4gcm93c10pKSwKICAgICAgICAicm93cyI6IHJvd3Ms',
    'CiAgICB9CiAgICAob3V0cHV0IC8gInBhcmV0b19pbmZvcm1hdGlvbl9zdW1tYXJ5Lmpzb24iKS53cml0ZV90ZXh0KAogICAg',
    'ICAgIGpzb24uZHVtcHMoc3VtbWFyeSwgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiKQogICAgcmV0dXJuIHN1bW1hcnkK',
    'CgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAg',
    'cGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdXRwdXQiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRk',
    'X2FyZ3VtZW50KCItLXdvcmxkLWNvdW50IiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTAyNCkKICAgIHBhcnNlci5hZGRfYXJndW1l',
    'bnQoIi0tdS1jb3VudCIsIHR5cGU9aW50LCBkZWZhdWx0PTEwMSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0taG9yaXpv',
    'biIsIHR5cGU9aW50LCBkZWZhdWx0PTEyOCkKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCiAgICByZXN1bHQgPSBy',
    'dW4oYXJncy5vdXRwdXQsIGFyZ3Mud29ybGRfY291bnQsIGFyZ3MudV9jb3VudCwgYXJncy5ob3Jpem9uKQogICAgcHJpbnQo',
    'anNvbi5kdW1wcyh7azogdiBmb3IgaywgdiBpbiByZXN1bHQuaXRlbXMoKSBpZiBrICE9ICJyb3dzIn0sIGluZGVudD0yKSkK',
]))
PLOT_SOURCE = base64.b64decode("".join([
    'IiIiUGxvdCB0aGUgY29udHJvbGxlZCBUSE0tMDYvMDcgbnVtZXJpY2FsIGNoZWNrIGZyb20gaXRzIENTViBvdXRwdXQuIiIi',
    'CgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGNzdgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBtYXRwbG90bGli',
    'LnB5cGxvdCBhcyBwbHQKCgpkZWYgcnVuKHRhYmxlOiBQYXRoLCBvdXRwdXQ6IFBhdGgpIC0+IE5vbmU6CiAgICB3aXRoIHRh',
    'YmxlLm9wZW4obmV3bGluZT0iIiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgaGFuZGxlOgogICAgICAgIHJvd3MgPSBsaXN0KGNz',
    'di5EaWN0UmVhZGVyKGhhbmRsZSkpCiAgICBmaWcsIGF4ZXMgPSBwbHQuc3VicGxvdHMoMSwgMiwgZmlnc2l6ZT0oOSwgMy41',
    'KSwgY29uc3RyYWluZWRfbGF5b3V0PVRydWUpCiAgICBzZWxlY3RlZCA9IHNvcnRlZCgocm93IGZvciByb3cgaW4gcm93cyBp',
    'ZiBmbG9hdChyb3dbImFtcGxpdHVkZSJdKSA9PSAxLjApLAogICAgICAgICAgICAgICAgICAgICAga2V5PWxhbWJkYSByb3c6',
    'IGludChyb3dbIksiXSkpCiAgICBidWRnZXRzID0gW2ludChyb3dbIksiXSkgZm9yIHJvdyBpbiBzZWxlY3RlZF0KICAgIGF4',
    'ZXNbMF0ubG9nbG9nKGJ1ZGdldHMsIFtmbG9hdChyb3dbIm1lYW5fZXhjZXNzX292ZXJfb3JhY2xlX2dyaWQiXSkgZm9yIHJv',
    'dyBpbiBzZWxlY3RlZF0sCiAgICAgICAgICAgICAgICAgICAiby0iLCBsYWJlbD0iTWVhc3VyZWQgZXhjZXNzLCBibGluZCBn',
    'cmlkIikKICAgIGF4ZXNbMF0ubG9nbG9nKGJ1ZGdldHMsIFtmbG9hdChyb3dbInRoZW9yZW1fZXhjZXNzX2xvd2VyX2JvdW5k',
    'Il0pIGZvciByb3cgaW4gc2VsZWN0ZWRdLAogICAgICAgICAgICAgICAgICAgInMtLSIsIGxhYmVsPSJUaGVvcmVtIGxvd2Vy',
    'IGJvdW5kIikKICAgIGF4ZXNbMF0ubG9nbG9nKGJ1ZGdldHMsIFtmbG9hdChyb3dbInRoZW9yZW1fdW5pZm9ybV91cHBlcl9i',
    'b3VuZCJdKSBmb3Igcm93IGluIHNlbGVjdGVkXSwKICAgICAgICAgICAgICAgICAgICJeOiIsIGxhYmVsPSJUaGVvcmVtIHVw',
    'cGVyIGJvdW5kIikKICAgIGF4ZXNbMF0uc2V0KHhsYWJlbD0iT3V0cHV0IGJ1ZGdldCBLIiwgeWxhYmVsPSJFeGNlc3MgY292',
    'ZXJhZ2UgcGVyIHJvdW5kIiwKICAgICAgICAgICAgICAgIHRpdGxlPSJCdWRnZXQgc2NhbGluZyAoYT0xKSIpCiAgICBheGVz',
    'WzBdLnNldF94dGlja3MoYnVkZ2V0cywgbGFiZWxzPVtzdHIoaykgZm9yIGsgaW4gYnVkZ2V0c10pCiAgICBheGVzWzBdLmxl',
    'Z2VuZChmcmFtZW9uPUZhbHNlLCBmb250c2l6ZT04KQogICAgc2VsZWN0ZWQgPSBzb3J0ZWQoKHJvdyBmb3Igcm93IGluIHJv',
    'd3MgaWYgaW50KHJvd1siSyJdKSA9PSA0KSwKICAgICAgICAgICAgICAgICAgICAgIGtleT1sYW1iZGEgcm93OiBmbG9hdChy',
    'b3dbImFtcGxpdHVkZSJdKSkKICAgIGFtcGxpdHVkZXMgPSBbZmxvYXQocm93WyJhbXBsaXR1ZGUiXSkgZm9yIHJvdyBpbiBz',
    'ZWxlY3RlZF0KICAgIGF4ZXNbMV0ucGxvdChhbXBsaXR1ZGVzLCBbZmxvYXQocm93WyJtZWFuX2V4Y2Vzc19vdmVyX29yYWNs',
    'ZV9ncmlkIl0pIGZvciByb3cgaW4gc2VsZWN0ZWRdLAogICAgICAgICAgICAgICAgICJvLSIsIGxhYmVsPSJNZWFzdXJlZCBl',
    'eGNlc3MsIGJsaW5kIGdyaWQiKQogICAgYXhlc1sxXS5wbG90KGFtcGxpdHVkZXMsIFtmbG9hdChyb3dbInRoZW9yZW1fZXhj',
    'ZXNzX2xvd2VyX2JvdW5kIl0pIGZvciByb3cgaW4gc2VsZWN0ZWRdLAogICAgICAgICAgICAgICAgICJzLS0iLCBsYWJlbD0i',
    'VGhlb3JlbSBsb3dlciBib3VuZCIpCiAgICBheGVzWzFdLnBsb3QoYW1wbGl0dWRlcywgW2Zsb2F0KHJvd1sidGhlb3JlbV91',
    'bmlmb3JtX3VwcGVyX2JvdW5kIl0pIGZvciByb3cgaW4gc2VsZWN0ZWRdLAogICAgICAgICAgICAgICAgICJeOiIsIGxhYmVs',
    'PSJUaGVvcmVtIHVwcGVyIGJvdW5kIikKICAgIGF4ZXNbMV0uc2V0KHhsYWJlbD0iSW50ZXJ2YWwgd2lkdGggYSIsIHlsYWJl',
    'bD0iRXhjZXNzIGNvdmVyYWdlIHBlciByb3VuZCIsCiAgICAgICAgICAgICAgICB0aXRsZT0iV2lkdGggc2NhbGluZyAoSz00',
    'KSIpCiAgICBheGVzWzFdLmxlZ2VuZChmcmFtZW9uPUZhbHNlLCBmb250c2l6ZT04KQogICAgZm9yIGF4aXMgaW4gYXhlczoK',
    'ICAgICAgICBheGlzLmdyaWQoYWxwaGE9MC4yNSkKICAgIG91dHB1dC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlz',
    'dF9vaz1UcnVlKQogICAgZmlnLnNhdmVmaWcob3V0cHV0LCBkcGk9MjAwKQogICAgZmlnLnNhdmVmaWcob3V0cHV0LndpdGhf',
    'c3VmZml4KCIucGRmIikpCiAgICBwbHQuY2xvc2UoZmlnKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBwYXJz',
    'ZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXRhYmxlIiwgdHlwZT1Q',
    'YXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdXRwdXQiLCB0eXBlPVBhdGgsIHJlcXVp',
    'cmVkPVRydWUpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgcnVuKGFyZ3MudGFibGUsIGFyZ3Mub3V0cHV0',
    'KQo=',
]))
PROOF_SOURCE = base64.b64decode("".join([
    'IyBUSE0tMDY6IGFuIGludGVydmFsLWluZm9ybWF0aW9uIHByaWNlIGZvciBmaW5pdGUgUGFyZXRvLXNldCBvdXRwdXQKCioq',
    'U3RhdHVzOiBQUk9WRUQgZm9yIHRoZSBzdGF0ZWQgYmlvYmplY3RpdmUgcXVhZHJhdGljIGNsYXNzIGFuZCBpbnRlcnZhbC1v',
    'bmx5IGZlZWRiYWNrLioqIFRoaXMgcmVzdWx0IGlzIGFib3V0IGEgKnNldCogb2YgYXQgbW9zdCBgS2AgYWN0aW9ucyBwZXIg',
    'cm91bmQgYW5kIGEgZ2VudWluZWx5IGNvbmZsaWN0aW5nIHR3by1vYmplY3RpdmUgUGFyZXRvIGludGVydmFsLiBJdCBkb2Vz',
    'IG5vdCBjbGFpbSBhIG1hdGNoaW5nIHJhdGUgZm9yIGFsbCBwcm9ibGVtcyBzYXRpc2Z5aW5nIEE34oCTQTEwLgoKTGV0IGBY',
    'PVswLDFdYCwgYG09MmAsIGBLLFTiiaUxYCwgYGE+MGAsIGFuZCBgcj0xLzIwYC4gVGhlIHVua25vd24gd29ybGQgcGFyYW1l',
    'dGVyIGlzIGDOuOKIiFtyLDEtcl1gLiBBdCBldmVyeSByb3VuZCBkZWZpbmUgdGhlIGxhdGVudCBvYmplY3RpdmVzCgpgaF8x',
    'Xs64KHgpPWEoeC3OuCtyKV4yYCwgIGBoXzJezrgoeCk9YSh4Lc64LXIpXjJgLgoKVGhlIGxlYXJuZXIgcmVjZWl2ZXMgb25s',
    'eSB0aGUgdHdvIGNvbXBsZXRlIGludGVydmFsLXZhbHVlZCBvYmplY3RpdmUgZnVuY3Rpb25zIGBGX2ooeCk9WzAsYV1gIChh',
    'bmQgYW55IGVuZHBvaW50L21pZHBvaW50IGdyYWRpZW50cyk7IHRoZXNlIGFyZSBpZGVudGljYWwgaW4gYWxsIHdvcmxkcy4g',
    'SXQgb3V0cHV0cyBhIHNldCBgU1904oqCWGAgb2YgY2FyZGluYWxpdHkgYXQgbW9zdCBgS2AgYmVmb3JlIGZlZWRiYWNrLiBM',
    'ZXQgYEVfzrg9W864LXIszrgrcl1gIGFuZCB1c2UgZXhhY3RseSBUSE0tMDUncyBsYXRlbnQgc2V0LWNvdmVyYWdlIGxvc3MK',
    'CmDOtV/OuChTKT1zdXBfe3XiiIhFX864fSBpbmZfe3jiiIhTfSBtYXhfe2o9MSwyfVtoX2pezrgoeCktaF9qXs64KHUpXV8r',
    'YC4KCkxldCBgzrVfS14qKM64KT1pbmZfe3xTfOKJpEt9IM61X864KFMpYCBiZSB0aGUgd29ybGQtc3BlY2lmaWMgb2ZmbGlu',
    'ZSBgS2AtcG9pbnQgYXBwcm94aW1hdGlvbiBlcnJvci4gRm9yICoqZXZlcnkqKiByYW5kb21pemVkIGludGVydmFsLW9ubHkg',
    'bGVhcm5lciB0aGVyZSBleGlzdHMgYSBmaXhlZCB3b3JsZCBgzrhgIHN1Y2ggdGhhdAoKYEVfYWxnIM6jX3t0PTF9XlQgW861',
    'X864KFNfdCktzrVfS14qKM64KV0g4omlIGFULyg4MEspID0gVV9ULyg4MEspYCwgICAoMSkKCndoZXJlIHRoZSB1bmlmb3Jt',
    'IHdlaWdodGVkIGludGVydmFsLXdpZHRoIHF1YW50aXR5IGlzIGBVX1Q9YVRgLiBJbiBwYXJ0aWN1bGFyLCBldmVuIGFmdGVy',
    'IHN1YnRyYWN0aW5nIHRoZSBmaW5pdGUtYEtgIGFwcHJveGltYXRpb24gZmxvb3IsIGludGVydmFsLW9ubHkgZmVlZGJhY2sg',
    'aW1wb3NlcyBhIGxpbmVhci1pbi13aWR0aCBQYXJldG8tc2V0IGNvdmVyYWdlIHBlbmFsdHkuIFRoaXMgZmFtaWx5IGhhcyBu',
    'byB0ZW1wb3JhbCBkcmlmdC4gQm90aCBsYXRlbnQgb2JqZWN0aXZlcyBhcmUgYDJhYC1zdHJvbmdseSBjb252ZXggYW5kIGAy',
    'YWAtTGlwc2NoaXR6IG9uIGBYYDsgdGhlaXIgbWluaW1pemVycyBhcmUgZGlzdGluY3QgYW5kIHRoZWlyIGVmZmljaWVudCBz',
    'ZXQgaXMgYSBub25kZWdlbmVyYXRlIGludGVydmFsLiBUaGUgZW5kcG9pbnQgZnVuY3Rpb25zIGFyZSBjb25zdGFudCBhbmQg',
    'Y29udmV4LgoKIyMgUHJvb2YKClRoZSB0d28gb2JqZWN0aXZlIG1pbmltYSBhcmUgYM64LXJgIGFuZCBgzrgrcmAuIEZvciBg',
    'eDzOuC1yYCwgbW92aW5nIHRvIGDOuC1yYCByZWR1Y2VzIGJvdGggb2JqZWN0aXZlczsgc2ltaWxhcmx5IGZvciBgeD7OuCty',
    'YC4gV2l0aGluIGBbzrgtcizOuCtyXWAsIGluY3JlYXNpbmcgYHhgIGluY3JlYXNlcyB0aGUgZmlyc3Qgb2JqZWN0aXZlIGFu',
    'ZCBkZWNyZWFzZXMgdGhlIHNlY29uZC4gSGVuY2UgdGhlIFBhcmV0by1lZmZpY2llbnQgZGVjaXNpb24gc2V0IGlzIGV4YWN0',
    'bHkgYEVfzrhgLgoKRm9yIGFueSBgeOKIiFhgLCBwdXQgYGQ9eC3OuGAuIEF0IHRoZSBlZmZpY2llbnQgdGFyZ2V0IGB1Pc64',
    'YCwgZGlyZWN0IGV4cGFuc2lvbiB5aWVsZHMKCmBtYXhfaiBbaF9qXs64KHgpLWhfal7OuCjOuCldXysgPSBhKGReMisycnxk',
    'fClgLgoKVGh1cyBgzrVfzrgoUyniiaUyYXIgZGlzdCjOuCxTKWAuIExldCBgzphgIGJlIHVuaWZvcm0gb24gdGhlIGludGVy',
    'dmFsIGBJPVtyLDEtcl1gIG9mIGxlbmd0aCBgTD0xLTJyYC4gQmVjYXVzZSBldmVyeSBmZWVkYmFjayBpbnRlcnZhbCBhbmQg',
    'ZW5kcG9pbnQgZ3JhZGllbnQgaXMgaWRlbnRpY2FsIGluIGFsbCB3b3JsZHMsIGBTX3RgIGlzIGluZGVwZW5kZW50IG9mIGDO',
    'mGAsIGV2ZW4gd2hlbiB0aGUgbGVhcm5lciB1c2VzIHJhbmRvbW5lc3MgYW5kIHBhc3QgZmVlZGJhY2suIEZvciBhbnkgZml4',
    'ZWQgc2V0IGBTYCB3aXRoIGF0IG1vc3QgYEtgIHBvaW50cywgdGhlIG9uZS1kaW1lbnNpb25hbCBuZWFyZXN0LW5laWdoYm9y',
    'IHF1YW50aXphdGlvbiBpbmVxdWFsaXR5IGdpdmVzCgpgRV/OmCBkaXN0KM6YLFMpIOKJpSBMLyg0SylgLgoKSW5kZWVkLCBp',
    'bnRlcnNlY3QgdGhlIFZvcm9ub2kgY2VsbHMgb2YgYFNgIHdpdGggYElgLiBJZiB0aGVpciBsZW5ndGhzIGFyZSBg4oSTXzEs',
    'Li4uLOKEk19xYCwgYHHiiaRLYCwgdGhlbiB0aGUgaW50ZWdyYWwgb2YgZGlzdGFuY2UgdG8gZWFjaCBjZWxsJ3MgY2VudGVy',
    'IGlzIGF0IGxlYXN0IGDihJNfacKyLzRgOyBDYXVjaHnigJNTY2h3YXJ6IGdpdmVzIGDOo19pIOKEk19pwrLiiaVMwrIvS2Au',
    'IEl0IGZvbGxvd3MgdGhhdAoKYEVfe86YLGFsZ30gzrVfzpgoU190KSDiiaUgYXJMLygySylgIGZvciBldmVyeSByb3VuZC4g',
    'ICAoMikKCkZvciBhbiBvcmFjbGUgdGhhdCBrbm93cyBgzrhgLCBwbGFjZSBgS2AgcG9pbnRzIGF0IHRoZSBtaWRwb2ludHMg',
    'b2YgYEtgIGVxdWFsIHN1YmludGVydmFscyBvZiBgRV/OuGAuIEV2ZXJ5IGB14oiIRV/OuGAgaGFzIGEgcG9pbnQgYHhgIGF0',
    'IGRpc3RhbmNlIGF0IG1vc3QgYHIvS2AuIEZvciBlaXRoZXIgb2JqZWN0aXZlLCBib3RoIGB4YCBhbmQgYHVgIGFyZSB3aXRo',
    'aW4gYDJyYCBvZiBpdHMgbWluaW1pemVyLCBzbwoKYFtoX2pezrgoeCktaF9qXs64KHUpXV8rIOKJpCA0YXJ8eC11fCDiiaQg',
    'NGFywrIvS2AuCgpDb25zZXF1ZW50bHkgYM61X0teKijOuCniiaQ0YXLCsi9LYC4gU3VidHJhY3QgdGhpcyB1bmlmb3JtIG9y',
    'YWNsZSB1cHBlciBib3VuZCBmcm9tICgyKSwgc3VtIG92ZXIgYFRgLCBhbmQgdXNlIGByPTEvMjBgLCBgTD05LzEwYDoKCmBh',
    'ckwvKDJLKS00YXLCsi9LID0gYSg5LzQwMC0xLzEwMCkvSyA9IGEvKDgwSylgLgoKVGhlIGF2ZXJhZ2Ugb3ZlciBgzphgIG9m',
    'IGV4cGVjdGVkIGV4Y2VzcyBjb3ZlcmFnZSBpcyBhdCBsZWFzdCB0aGlzIGFtb3VudCwgc28gc29tZSBmaXhlZCBgzrhgIGF0',
    'dGFpbnMgaXQuIEZpbmFsbHksIGV2ZXJ5IGludGVydmFsIGhhcyB3aWR0aCBgYWAsIGFuZCBzaW1wbGV4IHdlaWdodHMgc3Vt',
    'IHRvIG9uZSwgZ2l2aW5nIGBVX1Q9YVRgLiDilqEKCiMjIEluZm9ybWF0aW9uLW1vZGVsIGFuZCBzY29wZSBjaGVja3MKCi0g',
    'VGhlIGV4YWN0IGxhdGVudCBmdW5jdGlvbnMgYXJlICoqbm90KiogcmV2ZWFsZWQgYWZ0ZXIgYSByb3VuZC4gRnVsbCBrbm93',
    'bGVkZ2Ugb2YgaW50ZXJ2YWwgZW5kcG9pbnRzIGFuZCB0aGVpciBncmFkaWVudHMgc3RpbGwgcmV2ZWFscyBub3RoaW5nIGFi',
    'b3V0IGDOuGAuIElmIGxhdGVudCB2YWx1ZXMgb3IgZ3JhZGllbnRzIHdlcmUgc3VwcGxpZWQsIHRoaXMgaW5kaXN0aW5ndWlz',
    'aGFiaWxpdHkgYXJndW1lbnQgd291bGQgZmFpbC4KLSBUaGUgd2VpZ2h0ZWQgbWluaW1pemVyIGlzIGB1X864KHcpPc64K3Io',
    'd18yLXdfMSlgLCBzbyBldmVyeSBwb2ludCBvZiBgRV/OuGAgaXMgcmVwcmVzZW50ZWQgYW5kIHRoZSBtYXAgaXMgYHJgLUxp',
    'cHNjaGl0eiB1bmRlciBg4oSTXzFgIGRpc3RhbmNlIG9uIHRoZSBzaW1wbGV4LiBUaGUgY29uc3RydWN0aW9uIHRoZXJlZm9y',
    'ZSBzYXRpc2ZpZXMgdGhlIHNhbWUgZnJvbnQtcmVwcmVzZW50YXRpb24gYW5kIHN0cm9uZy1jb252ZXhpdHkgZ2VvbWV0cnkg',
    'dXNlZCBieSBUSE0tMDUuIEl0IHBlcm1pdHMgYW55IHNldC1vdXRwdXQgYWxnb3JpdGhtIHdpdGggYXQgbW9zdCBgS2AgcG9p',
    'bnRzLCBub3QganVzdCBhIHByZWZlcmVuY2UgZ3JpZC4KLSBFcXVhdGlvbiAoMSkgaXMgYSAqKmxvd2VyIGJvdW5kIG9uIGV4',
    'Y2VzcyBzZXQtY292ZXJhZ2UgbG9zcyBvdmVyIHRoZSBiZXN0IHdvcmxkLXNwZWNpZmljIGBLYC1wb2ludCBzZXQqKiwgbm90',
    'IG9uIGEgc2luZ2xlLXBsYXkgUGFyZXRvIGdhcCwgaHlwZXJ2b2x1bWUsIG9yIGEgdW5pdmVyc2FsIG1pbmltYXggcmF0ZS4g',
    'VEhNLTA1J3MgY3VycmVudCB1cHBlciBib3VuZCBkb2VzIG5vdCBtYXRjaCB0aGUgYDEvS2AgZGVwZW5kZW5jZS4KLSBXaGVu',
    'IGludGVydmFsIHdpZHRoIHNocmlua3MgdGhyb3VnaCBgYeKGkjBgLCB0aGUgYm91bmQgdmFuaXNoZXMuIFRoZSBzY2FsaW5n',
    'IGFsc28gY2hhbmdlcyB0aGUgbGF0ZW50IExpcHNjaGl0eiBhbmQgc3Ryb25nLWNvbnZleGl0eSBjb25zdGFudHM7IGl0IGRv',
    'ZXMgbm90IGlzb2xhdGUgd2lkdGggd2hpbGUga2VlcGluZyB0aG9zZSBjb25zdGFudHMgZml4ZWQuIEEgY29tbW9uLWNsYXNz',
    'IHdpZHRoIGNvbXBhcmlzb24gd2l0aCBmaXhlZCByZWd1bGFyaXR5IGNvbnN0YW50cyByZW1haW5zIG9wZW4uCi0gSW4gdGhl',
    'IGV4YWN0LWxhdGVudC1mZWVkYmFjayB2YXJpYW50LCBvbmUgb2JzZXJ2YXRpb24gb2YgdGhlIGZ1bGwgb2JqZWN0aXZlcyBp',
    'ZGVudGlmaWVzIGDOuGAsIGFsbG93aW5nIHdvcmxkLXNwZWNpZmljIGBLYC1wb2ludCBvdXRwdXQgZnJvbSB0aGUgc2Vjb25k',
    'IHJvdW5kIG9ud2FyZC4gVGhpcyBleHBsYWlucyB0aGUgaW5mb3JtYXRpb24gb3JpZ2luIG9mIHRoZSBsaW5lYXItaW4tYFRg',
    'IGV4Y2VzcyB1bmRlciBpbnRlcnZhbC1vbmx5IGZlZWRiYWNrLgo=',
]))
PROOF_UPPER_SOURCE = base64.b64decode("".join([
    'IyBUSE0tMDc6IG1hdGNoaW5nIG9yZGVyIGZvciB0aGUgVEhNLTA2IGhpZGRlbiBQYXJldG8gZmFtaWx5CgoqKlN0YXR1czog',
    'UFJPVkVEIGZvciBleGFjdGx5IHRoZSBmYW1pbHkgYW5kIGludGVydmFsLW9ubHkgcHJvdG9jb2wgb2YgVEhNLTA2LioqIExl',
    'dCBgTV97SyxUfWAgYmUgdGhlIG1pbmltYXggZXhwZWN0ZWQgKmV4Y2VzcyogY3VtdWxhdGl2ZSBzZXQtY292ZXJhZ2UgbG9z',
    'cywKCmBNX3tLLFR9PWluZl9BIHN1cF97zrjiiIhbciwxLXJdfSBFX0EgzqNfdFvOtV/OuChTX3QpLc61X0teKijOuCldYCwK',
    'CndoZXJlIGBBYCByYW5nZXMgb3ZlciByYW5kb21pemVkIGludGVydmFsLW9ubHkgYWxnb3JpdGhtcyB3aXRoIGF0IG1vc3Qg',
    'YEtgIG91dHB1dCBwb2ludHMgcGVyIHJvdW5kLCBgcj0xLzIwYCwgYW5kIHRoZSB0d28gb2JqZWN0aXZlcyBhbmQgY292ZXJh',
    'Z2UgbWV0cmljIGFyZSBhcyBpbiBUSE0tMDYuIFRoZW4KCmBVX1QvKDgwSykg4omkIE1fe0ssVH0g4omkIFVfVC9LYCwgd2l0',
    'aCBgVV9UPWFUYC4KClRodXMgdGhlIG1pbmltYXggaW5mb3JtYXRpb24gcHJpY2UgZm9yIHRoaXMgKipyZXN0cmljdGVkIGhp',
    'ZGRlbi13b3JsZCBmYW1pbHkqKiBoYXMgZXhhY3Qgb3JkZXIgYM6YKFVfVC9LKWAuIFRoaXMgaXMgbm90IGEgZ2VuZXJhbCBB',
    'N+KAk0ExMCBtYXRjaGluZyB1cHBlciBib3VuZCwgbm9yIGEgbG93ZXIgYm91bmQgd2l0aCBmaXhlZCByZWd1bGFyaXR5IGNv',
    'bnN0YW50cyBhcyB3aWR0aCB2YXJpZXMuCgojIyBQcm9vZgoKVEhNLTA2IGdpdmVzIHRoZSBsb3dlciBpbmVxdWFsaXR5IGZv',
    'ciBldmVyeSByYW5kb21pemVkIGFsZ29yaXRobSwgaGVuY2UgZm9yIHRoZSBpbmZpbXVtIG92ZXIgYWxnb3JpdGhtcy4KCkZv',
    'ciB0aGUgdXBwZXIgaW5lcXVhbGl0eSwgdXNlIHRoZSBkZXRlcm1pbmlzdGljLCBmZWVkYmFjay1pbmRlcGVuZGVudCBtaWRw',
    'b2ludCBncmlkIGBTPXsgKGkrMS8yKS9LIDogaT0wLC4uLixLLTEgfWAgYXQgZXZlcnkgcm91bmQuIEZvciBldmVyeSBgdeKI',
    'iEVfzrjiioJbMCwxXWAsIHNvbWUgYHjiiIhTYCBzYXRpc2ZpZXMgYHx4LXV84omkMS8oMkspYC4gQm90aCBxdWFkcmF0aWMg',
    'b2JqZWN0aXZlcyBgaF9qXs64YCBhcmUgYDJhYC1MaXBzY2hpdHogb24gYFswLDFdYDogdGhlaXIgbWluaW1pemVycyBsaWUg',
    'aW4gYFswLDFdYCwgc28gYHwoaF9qXs64KScoeil8PTJhfHotY19qfOKJpDJhYC4gSGVuY2UKCmBtYXhfaltoX2pezrgoeCkt',
    'aF9qXs64KHUpXV8rIOKJpCAyYXx4LXV8IOKJpCBhL0tgLgoKVGFraW5nIHRoZSBpbmZpbXVtIG92ZXIgYFNgIGFuZCBzdXBy',
    'ZW11bSBvdmVyIGB1YCB5aWVsZHMgYM61X864KFMp4omkYS9LYC4gU2luY2UgYM61X0teKijOuCniiaUwYCwgZXhjZXNzIGlz',
    'IGF0IG1vc3QgYGEvS2AgZWFjaCByb3VuZC4gU3VtbWluZyBvdmVyIGBUYCBnaXZlcyBgYVQvSz1VX1QvS2AsIHVuaWZvcm1s',
    'eSBpbiBgzrhgLiBUaGlzIGZlYXNpYmxlIGFsZ29yaXRobSBwcm92ZXMgdGhlIG1pbmltYXggdXBwZXIgaW5lcXVhbGl0eS4g',
    '4pahCgojIyBJbnRlcnByZXRhdGlvbgoKLSBUaGUgdXBwZXIgYWxnb3JpdGhtIG91dHB1dHMgYEtgIGFjdGlvbnMgYW5kIG5l',
    'ZWRzIG5vIGxhdGVudCBvYnNlcnZhdGlvbnMgb3IgZmVlZGJhY2sgcXVlcmllcy4gSXRzIGNvdmVyYWdlIGlzIHVuaWZvcm1s',
    'eSBib3VuZGVkIGV2ZW4gdGhvdWdoIHRoZSB3b3JsZCBpcyBuZXZlciBpZGVudGlmaWVkLgotIFRoZSBmYWN0b3ItODAgZ2Fw',
    'IGlzIGluIGNvbnN0YW50cywgbm90IG9yZGVyLiBUaGUgY3VycmVudCBwcm9vZiBkb2VzIG5vdCBpZGVudGlmeSB0aGUgZXhh',
    'Y3QgbWluaW1heCBjb25zdGFudCBvciBvcHRpbWFsIHJvYnVzdCBncmlkLgotIFRoaXMgbWF0Y2hpbmcgb3JkZXIgdXNlcyB0',
    'aGUgZmFjdCB0aGF0IHRoZSBlbnRpcmUgc2V0IG9mIHBvc3NpYmxlIGVmZmljaWVudCB0YXJnZXRzIGlzIGEgb25lLWRpbWVu',
    'c2lvbmFsIGludGVydmFsIG9mIGZpeGVkIGRpYW1ldGVyLiBJbiBtb3JlIGdlbmVyYWwgZnJvbnRzLCBjb3ZlcmluZyBudW1i',
    'ZXJzIGFuZCBvYmplY3RpdmUgTGlwc2NoaXR6IGNvbnN0YW50cyBkZXRlcm1pbmUgdGhlIHVwcGVyIHJhdGU7IFRITS0wNSdz',
    'IGBLYCBkZXBlbmRlbmNlIHJlbWFpbnMgbG9vc2UuCg==',
]))
COMPARISON_SOURCE = base64.b64decode("".join([
    'IyBDbG9zZXN0LXJlc3VsdCBjb21wYXJpc29uIGZvciB0aGUgSkNBTSBkcmFmdAoKKipTdGF0dXM6IFBBUlRJQUwgRlVMTC1U',
    'RVhUIEFVRElULioqIFZlcmlmaWVkIGZyb20gcHJpbWFyeSBwdWJsaXNoZXIvYXV0aG9yIHNvdXJjZXMgb24gMjAyNi0xMC0w',
    'My4gVGhlIGZ1bGwgWWFkYXbigJNWZXRyaXZlbCBhcnRpY2xlIGFuZCBTYXZhcnnigJNXaWVjZWsgY2hhcHRlciB3ZXJlIGlu',
    'YWNjZXNzaWJsZSB0byB0aGlzIGF1ZGl0LCBhbmQgdGhlIHVzZXIgaGFzIG5vIGNvcGllcy4gVGhlaXIgZXhhY3QgdGhlb3Jl',
    'bSBhc3N1bXB0aW9ucyBhbmQgY29uc3RhbnRzIGFyZSB0aGVyZWZvcmUgKip1bnZlcmlmaWVkKiouIE5vIG9yaWdpbmFsaXR5',
    'IG9yIHByaW9yaXR5IGNsYWltIHJlc3RzIG9uIHRob3NlIG1pc3NpbmcgdGV4dHMuCgp8IFdvcmsgfCBGZWVkYmFjayAvIG9y',
    'YWNsZSB8IENvbXBhcmF0b3IgYW5kIHJlZ3JldCB8IEFjdGlvbi9xdWVyeSBidWRnZXQgfCBWZXJpZmllZCByZXN1bHQgfCBS',
    'ZWxhdGlvbiB0byBKQ0FNIFRITS0wMiwgVEhNLTA1LCBUSE0tMDYgfAp8LS0tfC0tLXwtLS18LS0tfC0tLXwtLS18CnwgW1lh',
    'ZGF24oCTVmV0cml2ZWwgKDIwMjUgb25saW5lOyAyMDI2IGlzc3VlKV0oaHR0cHM6Ly9kb2kub3JnLzEwLjEwODAvMDAyMDcx',
    'NjAuMjAyNS4yNTkzOTI4KSB8IEFic3RyYWN0IHN0YXRlcyBncmFkaWVudHMgb2YgaW50ZXJ2YWwtdmFsdWVkIGNvbnZleCBv',
    'YmplY3RpdmVzOyBleGFjdCBvcmFjbGUgdW5zcGVjaWZpZWQgaW4gYWNjZXNzaWJsZSBhYnN0cmFjdC4gfCBJbnRlcnZhbC12',
    'YWx1ZWQgb25saW5lIGNvbnZleCByZWdyZXQ7IGV4YWN0IGNvbXBhcmF0b3IgYW5kIGludGVydmFsIG9yZGVyIHVudmVyaWZp',
    'ZWQuIHwgVW52ZXJpZmllZC4gfCBBYnN0cmFjdCBjbGFpbXMgYSBncmFkaWVudCBhbGdvcml0aG0gd2l0aCBzdWJsaW5lYXIg',
    'cmVncmV0LiB8IFBvdGVudGlhbGx5IGNsb3NlIHRvIFRITS0wMi4gVEhNLTAyIG11c3QgYmUgZnJhbWVkIGFyb3VuZCAqbGF0',
    'ZW50IGxvc3MgY29udGFpbmVkIGluIGludGVydmFscyosIGR5bmFtaWMgY29tcGFyYXRvciwgYW5kIHRoZSBleHBsaWNpdCB0',
    'd28tbG9jYXRpb24gd2lkdGggdHJhbnNmZXIsIHN1YmplY3QgdG8gZnVsbC10ZXh0IGNoZWNrLiBJdCBjYW5ub3QgYmUgY2Fs',
    'bGVkIHRoZSBmaXJzdCBvbmxpbmUgaW50ZXJ2YWwtcmVncmV0IHJlc3VsdC4gfAp8IFtKaWFuZyBldCBhbC4sIElDTFIgMjAy',
    'M10oaHR0cHM6Ly9vcGVucmV2aWV3Lm5ldC9wZGY/aWQ9ZEtrTW5DV2ZWbW0pIHwgRmlyc3Qtb3JkZXIgdmVjdG9yIG9iamVj',
    'dGl2ZSBncmFkaWVudHMsIGNvbXBvc2l0ZSBncmFkaWVudCBhbmQgbWlycm9yLWRlc2NlbnQgdXBkYXRlIGRlc2NyaWJlZCBp',
    'biB0aGUgYXV0aG9yIHBhcGVyJ3Mgc2VhcmNoYWJsZSB0ZXh0LiBObyBpbnRlcnZhbC1vbmx5IGhpZGRlbiBsYXRlbnQgd29y',
    'bGQgaXMgZGVzY3JpYmVkLiB8IFBhcmV0by1zdWJvcHRpbWFsaXR5LWdhcCBvbmxpbmUgbXVsdGlvYmplY3RpdmUgcmVncmV0',
    'OyBhIHNlYXJjaGFibGUgYXV0aG9yLW1hbnVzY3JpcHQgdmVyc2lvbiByZXBvcnRzIHN0YXRpYyBgTyhzcXJ0KFQpKWAgYW5k',
    'IGR5bmFtaWMgYE8oVl9UXnsxLzN9VF57Mi8zfSlgLiBUaGUgZmluYWwgY29uZmVyZW5jZSB0ZXh0IGNvdWxkIG5vdCBiZSBm',
    'ZXRjaGVkIGZvciB2ZXJzaW9uLWxldmVsIHZlcmlmaWNhdGlvbi4gfCBTaW5nbGUgZGVjaXNpb24gcGVyIHJvdW5kLCB3aXRo',
    'IG11bHRpcGxlIG9iamVjdGl2ZSBncmFkaWVudHMgdXNlZCBpbiB0aGUgdXBkYXRlIGFjY29yZGluZyB0byB0aGUgcGFwZXIn',
    'cyBhbGdvcml0aG0gZGVzY3JpcHRpb24uIHwgVGhlIHBhcGVyIGludHJvZHVjZXMgRFItT01NRCBhbmQgcmVncmV0IGd1YXJh',
    'bnRlZXMgZm9yIFBTRy1iYXNlZCBNTy1PQ08uIHwgT3VyIFRITS0wNSBtZWFzdXJlcyBwZXItcm91bmQgZmluaXRlLXNldCBj',
    'b3ZlcmFnZSBgc3VwX3UgaW5mX3ggbWF4X2pbLi4uXV8rYCwgdXNlcyBgS2Agb3V0cHV0cyBhbmQgZ3JhZGllbnRzLCBhbmQg',
    'aW5jbHVkZXMgbGF0ZW50IGludGVydmFsIGNvbnRhaW5tZW50LiBUSE0tMDYgaXMgYW4gaW5mb3JtYXRpb24gbG93ZXIgYm91',
    'bmQgZm9yIHRoaXMgc2V0IG1ldHJpYywgbm90IEppYW5nJ3Mgc2luZ2xlLWRlY2lzaW9uIFBTRyByZWdyZXQuIE5ldmVyIGNv',
    'bXBhcmUgdGhlIHJhdGVzIGFzIGlmIHRoZXkgd2VyZSB0aGUgc2FtZSBtZXRyaWMuIHwKfCBbU2F2YXJ54oCTV2llY2VrICgy',
    'MDIzKV0oaHR0cHM6Ly9kb2kub3JnLzEwLjEwMDcvOTc4LTMtMDMxLTUwMzMwLTZfMTQpLCBbYXV0aG9yIGFic3RyYWN0XSho',
    'dHRwczovL29yLmNsZW1zb24uZWR1L3dwLWNvbnRlbnQvdXBsb2Fkcy8yMDIzLzA2L01XaWVjZWs3SnVuZTIwMjMucGRmKSB8',
    'IEV4YWN0IGZlZWRiYWNrIGFuZCBvcmFjbGUgdW52ZXJpZmllZCBmcm9tIGFjY2Vzc2libGUgYWJzdHJhY3QuIHwgSHlwZXJ2',
    'b2x1bWUtYmFzZWQgc2V0IHJlZ3JldCByZWxhdGl2ZSB0byB0aGUgUGFyZXRvIHNldCBvZiBhbiBvZmZsaW5lIGFjY3VtdWxh',
    'dGVkIG91dGNvbWUgcHJvYmxlbSwgYWNjb3JkaW5nIHRvIGF1dGhvciBhYnN0cmFjdC4gfCBFeGFjdCBvdXRwdXQvcXVlcnkg',
    'YnVkZ2V0IHVudmVyaWZpZWQuIHwgQXV0aG9yIGFic3RyYWN0IHNheXMgdGhlIHNldCByZWdyZXQgaGFzIGFuIHVwcGVyIGJv',
    'dW5kIHJlZHVjaW5nIHRvIGEgc2NhbGFyIGJvdW5kLiB8IERpcmVjdCBwcmlvciBzZXQtcmVncmV0IHdvcmsuIE91ciBjb3Zl',
    'cmFnZSBtZXRyaWMgYW5kIHBlci1yb3VuZCBoaWRkZW4tbGF0ZW50IGZlZWRiYWNrIGFyZSBkaWZmZXJlbnQgb24gdGhlIGF2',
    'YWlsYWJsZSBkZXNjcmlwdGlvbiwgYnV0IGV4YWN0IHRoZW9yZW0gY29tcGFyaXNvbiBhd2FpdHMgZnVsbCB0ZXh0LiBObyBm',
    'aXJzdCBzZXQtcmVncmV0IGNsYWltLiB8CnwgW1h14oCTS2xhYmphbiwgSUNNTCAyMDIzXShodHRwczovL3Byb2NlZWRpbmdz',
    'Lm1sci5wcmVzcy92MjAyL3h1MjNpLmh0bWwpIHwgTXVsdGlhcm1lZCBiYW5kaXQgZmVlZGJhY2sgaW4gc3RvY2hhc3RpYy9h',
    'ZHZlcnNhcmlhbCByZWdpbWVzLiB8IFBhcmV0byByZWdyZXQgd2l0aG91dCBzY2FsYXJpemF0aW9uLiB8IEZpbml0ZSBhcm1z',
    'OyBzZXQtb3V0cHV0IGJ1ZGdldCBub3Qgb3VyIGBLYC1wb2ludCBjb252ZXgtZG9tYWluIHByb3RvY29sLiB8IFVwcGVyIGFu',
    'ZCBsb3dlciBQYXJldG8tcmVncmV0IGJvdW5kcyByZXBvcnRlZCBpbiB0aGUgUE1MUiBhYnN0cmFjdC4gfCBTaG93cyBQYXJl',
    'dG8gcmVncmV0IGFuZCBsb3dlciBib3VuZHMgYXJlIGVzdGFibGlzaGVkIHRvcGljczsgb3VyIGNvbnRpbnVvdXMgY29udmV4',
    'IGludGVydmFsLW9ubHkgc2V0LWNvdmVyYWdlIHRhcmdldCBkaWZmZXJzLiB8CnwgW0dyb2V0em5lcuKAk1dlcm5lciwgRUpP',
    'UiAyMDIyXShodHRwczovL2RvaS5vcmcvMTAuMTAxNi9qLmVqb3IuMjAyMS4wMy4wNjgpIHwgT2ZmbGluZSBzY2VuYXJpb3Mg',
    'aW4gdW5jZXJ0YWluIG11bHRpb2JqZWN0aXZlIG9wdGltaXphdGlvbiwgZnJvbSBqb3VybmFsIGFic3RyYWN0LiB8IE11bHRp',
    'dmFyaWF0ZSByb2J1c3QgKHJlbGF0aXZlKSByZWdyZXQuIHwgTm8gc2VxdWVudGlhbCBvbmxpbmUgZmVlZGJhY2sgZGVzY3Jp',
    'YmVkIGluIGFic3RyYWN0LiB8IFJvYnVzdC1yZWdyZXQgZnJhbWV3b3JrIG5vdCBsaW1pdGVkIHRvIGludGVydmFsIHVuY2Vy',
    'dGFpbnR5LiB8IEF2b2lkIGFueSBjbGFpbSBvZiBmaXJzdCB1bmNlcnRhaW50eS1hd2FyZSBtdWx0aXZhcmlhdGUgcmVncmV0',
    'LiBEaXN0aW5ndWlzaCB0aGUgb25saW5lIGZlZWRiYWNrIGluZm9ybWF0aW9uIGJhcnJpZXIuIHwKCiMjIENsYWltIGRpc2Np',
    'cGxpbmUKCjEuIFRITS0wMiBpcyBhIGNvcnJlY3QgZHluYW1pYy1yZWdyZXQgKipiYXNlbGluZSoqLCBub3QgYSBkZW1vbnN0',
    'cmF0ZWQgbm92ZWwgYWxnb3JpdGhtaWMgcmF0ZS4KMi4gVEhNLTA1IGlzIGEgZmluaXRlLXNldC1vdXRwdXQgKip1cHBlciBi',
    'b3VuZCB3aXRoIHN0cm9uZyByZXByZXNlbnRhdGlvbiBhbmQgYEtgLXF1ZXJ5IGFzc3VtcHRpb25zKiosIG5vdCBhIG9uZS1w',
    'bGF5IFBhcmV0byBndWFyYW50ZWUuIEl0cyBjdXJyZW50IGBLYCBkZXBlbmRlbmNlIGlzIGxvb3NlLgozLiBUSE0tMDYgcHJv',
    'dmVzIGEgcmVzdHJpY3RlZCBgzqkoVV9UL0spYCAqZXhjZXNzKiBzZXQtY292ZXJhZ2UgaW5mb3JtYXRpb24gYmFycmllciBp',
    'biBhIGNvbmZsaWN0aW5nIGJpb2JqZWN0aXZlIGZhbWlseS4gVGhlIHNlYXJjaGVkIHByaW1hcnkgc291cmNlcyBhYm92ZSBk',
    'aWQgbm90IHNob3cgdGhhdCBleGFjdCBzdGF0ZW1lbnQsIGJ1dCBhYnNlbmNlIGluIHRoaXMgZmluaXRlIHNlYXJjaCBpcyBu',
    'b3QgcHJvb2Ygb2YgcHJpb3JpdHkuCjQuIEEgam91cm5hbCBkcmFmdCBtYXkgcHJlc2VudCBUSE0tMDYgYXMgdGhlIHByaW5j',
    'aXBhbCBuZXcgKipyZXN1bHQgb2YgdGhpcyBwcm9qZWN0KiosIHdoaWxlIHN0YXRpbmcgdGhhdCBicm9hZGVyIG5vdmVsdHkg',
    'YW5kIGEgbWF0Y2hpbmcgY29tbW9uLWNsYXNzIHVwcGVyIHJhdGUgcmVtYWluIG9wZW4uIFJlY2hlY2sgdGhlIHR3byBtaXNz',
    'aW5nIGZ1bGwgdGV4dHMgYmVmb3JlIGEgc3VibWlzc2lvbiBjbGFpbS4K',
]))
MANUSCRIPT_SOURCE = b''
JOURNAL_PLAN_SOURCE = base64.b64decode("".join([
    'IyBKb3VybmFsIGNob2ljZSBhbmQgc3VibWlzc2lvbiBnYXRlIOKAlCAzIE9jdG9iZXIgMjAyNgoKIyMgUHJlZmVycmVkIHRh',
    'cmdldAoKKipbSm91cm5hbCBvZiBHbG9iYWwgT3B0aW1pemF0aW9uXShodHRwczovL3d3dy5zY2ltYWdvanIuY29tL2pvdXJu',
    'YWxzZWFyY2gucGhwP3E9ODgzNDUmdGlwPXNpZCkqKiBpcyB0aGUgcHJlZmVycmVkIFExIHRhcmdldCB1bmRlciB0aGUgU0NJ',
    'bWFnbyAyMDI0IGNsYXNzaWZpY2F0aW9uIGNvbnN1bHRlZCBkdXJpbmcgdGhpcyBhdWRpdC4gSXRzIHRoZW9yZXRpY2FsIGFu',
    'ZCBjb21wdXRhdGlvbmFsIG9wdGltaXphdGlvbiBzY29wZSBpcyBhIGNsb3NlciBmaXQgdG8gYSBQYXJldG8tc2V0IGluZm9y',
    'bWF0aW9uIGJvdW5kIHRoYW4gdGhlIGN1cnJlbnQgdmVyc2lvbiBvZiB0aGUgcHJvamVjdCBpcyB0byAqSm91cm5hbCBvZiBD',
    'b21wdXRhdGlvbmFsIGFuZCBBcHBsaWVkIE1hdGhlbWF0aWNzKiAoSkNBTSkuIFRoaXMgaXMgYSBzY29wZSBjaG9pY2UsIG5v',
    'dCBhIHByZWRpY3Rpb24gb2YgYWNjZXB0YW5jZTsgY29uZmlybSB0aGUgZXhhY3QgUTEgaW5kZXgsIHN1YmplY3QgY2F0ZWdv',
    'cnksIGFuZCB5ZWFyIHJlcXVpcmVkIGJ5IHRoZSBhdXRob3IncyBpbnN0aXR1dGlvbiBiZWZvcmUgc3VibWlzc2lvbi4KCioq',
    'W0pDQU1dKGh0dHBzOi8vc2hvcC5lbHNldmllci5jb20vam91cm5hbHMvam91cm5hbC1vZi1jb21wdXRhdGlvbmFsLWFuZC1h',
    'cHBsaWVkLW1hdGhlbWF0aWNzLzAzNzctMDQyNykqKiByZW1haW5zIGEgY29uZGl0aW9uYWwgYWx0ZXJuYXRpdmUgaWYgdGhl',
    'IHBhcGVyIGFjcXVpcmVzIGEgc3Vic3RhbnRpYWxseSBzdHJvbmdlciBjb21wdXRhdGlvbmFsIGNvbnRyaWJ1dGlvbiBvciB0',
    'aWdodGVyIGdlbmVyYWwgY292ZXJhZ2UgcmF0ZS4gSXRzIHB1Ymxpc2hlciBzY29wZSBhc2tzIGZvciBhIG5ldyBjb21wdXRh',
    'dGlvbmFsIHRlY2huaXF1ZSBvciBzaWduaWZpY2FudGx5IGltcHJvdmVkIGFuYWx5c2lzIHN1cHBvcnRlZCBieSBub250cml2',
    'aWFsIG51bWVyaWNhbCBleGFtcGxlcy4gVEhNLTA3IG5vdyBtYXRjaGVzIHRoZSBsb3dlciBib3VuZCdzIG9yZGVyIGZvciB0',
    'aGUgY29uc3RydWN0ZWQgZmFtaWx5LCBidXQgdGhlIG9uZS1kaW1lbnNpb25hbCBmYW1pbHkgYW5kIGxvb3NlIGdlbmVyYWwg',
    'VEhNLTA1IGJvdW5kIG1heSBzdGlsbCBiZSB0b28gbmFycm93IHRoZXJlLgoKIyMgUmVxdWlyZWQgc3VibWlzc2lvbiB3b3Jr',
    'CgoxLiBPYnRhaW4gdGhlIGZ1bGwgWWFkYXbigJNWZXRyaXZlbCBhcnRpY2xlIGFuZCBTYXZhcnnigJNXaWVjZWsgY2hhcHRl',
    'ciB0aHJvdWdoIGluc3RpdHV0aW9uYWwgYWNjZXNzIG9yIGF1dGhvcnMgYW5kIGNvbXBhcmUgdGhlIGV4YWN0IHRoZW9yZW0g',
    'YXNzdW1wdGlvbnMsIGZlZWRiYWNrIG9yYWNsZSwgY29tcGFyYXRvciwgbWV0cmljLCBhbmQgcXVlcnkgYnVkZ2V0IHdpdGgg',
    'VEhNLTAyLzA1LzA2LiBUaGUgY3VycmVudCBtYXRyaXggaXMgcGFydGlhbCBhbmQgbXVzdCBub3QgYmUgcmVwcmVzZW50ZWQg',
    'YXMgYSBmdWxsLXRleHQgY29tcGFyaXNvbi4KMi4gUHJvdmUgYSBtYXRjaGluZyBvciBuZWFyLW1hdGNoaW5nIHJhdGUgaW4g',
    'YSBicm9hZGVyIGNsYXNzIHdpdGggcmVndWxhcml0eSBjb25zdGFudHMgaGVsZCBmaXhlZCB3aGlsZSBpbnRlcnZhbCB3aWR0',
    'aCB2YXJpZXMsIG9yIHNoYXJwZW4gdGhlIFRITS0wNSBzZXQtY292ZXJhZ2UgYm91bmQgYW5kIGl0cyBkZXBlbmRlbmNlIG9u',
    'IGBLYC4gVEhNLTA2LzA3IGVzdGFibGlzaCBgzpgoVV9UL0spYCBtaW5pbWF4IGV4Y2VzcyBvbmx5IGZvciBvbmUgdHJhbnNs',
    'YXRlZCBxdWFkcmF0aWMgZmFtaWx5LgozLiBBZGQgb25lIG5vbmNvbnN0cnVjdGVkIGV4cGVyaW1lbnQgaW4gd2hpY2ggdGhl',
    'IGxhdGVudCBjb250YWlubWVudCBhc3N1bXB0aW9uIGlzIGp1c3RpZmllZCBvciBlbXBpcmljYWxseSBjYWxpYnJhdGVkLCBh',
    'bmQgbWVhc3VyZSB0aGUgc2FtZSBmaW5pdGUtc2V0IGNvdmVyYWdlIG1ldHJpYyB3aXRoIHRoZSBzYW1lIGZlZWRiYWNrL3F1',
    'ZXJ5IGJ1ZGdldC4gRVhQLTEwIEJhbmsgTWFya2V0aW5nIGlzIGFuIGFwcGxpY2F0aW9uIGRpYWdub3N0aWMgYnV0IGl0cyBv',
    'YnNlcnZhdGlvbmFsIGJhbmRzIGFyZSBub3QgYSB2ZXJpZmllZCBwb3B1bGF0aW9uIGNlcnRpZmljYXRlLgo0LiBVcGRhdGUg',
    'YmlibGlvZ3JhcGhpYyBtZXRhZGF0YSBhbmQgYXV0aG9yIGxpc3QsIHRoZW4gcmVjb21waWxlIGFuZCB2aXN1YWxseSBwcm9v',
    'ZnJlYWQgdGhlIG1hbnVzY3JpcHQuIFRoZSBDb2RleCBidWlsdC1pbiBjb21waWxlciB3YXMgdW5hdmFpbGFibGUgb24gdGhp',
    'cyBob3N0IChzdGFuZGFyZCBkaXJlY3RvcmllcyBlcnJvcikgb24gMyBPY3RvYmVyIDIwMjYsIHNvIFBERiBjb21waWxhdGlv',
    'biBpcyB1bnZlcmlmaWVkLgoKVGhlIGN1cnJlbnQgbWFudXNjcmlwdCBpcyBhbiBpbnRlcm5hbCByZXNlYXJjaCBkcmFmdCwg',
    'bm90IGEgc3VibWlzc2lvbi1yZWFkeSBRMSBhcnRpY2xlLgo=',
]))
if RUN_OK:
    (RUN_ROOT / "pareto_information_lower_bound.py").write_bytes(EXPERIMENT_SOURCE)
    (RUN_ROOT / "plot_pareto_information.py").write_bytes(PLOT_SOURCE)
    (RUN_ROOT / "proof_thm06.md").write_bytes(PROOF_SOURCE)
    (RUN_ROOT / "proof_thm07.md").write_bytes(PROOF_UPPER_SOURCE)
    (RUN_ROOT / "closest_theorem_comparison.md").write_bytes(COMPARISON_SOURCE)
    record("sources", "passed", "experiment, plotting code, two theorem proofs, comparison, paper draft")
else:
    record("sources", "skipped", "bootstrap failed")

## 3. Check the Pareto-information lower bound and plot direct scaling


In [ ]:
if RUN_OK:
    started = time.perf_counter()
    commands = [
        [sys.executable, str(RUN_ROOT / "pareto_information_lower_bound.py"),
         "--output", str(RUN_ROOT / "results")],
        [sys.executable, str(RUN_ROOT / "plot_pareto_information.py"),
         "--table", str(RUN_ROOT / "results" / "pareto_information_lower_bound.csv"),
         "--output", str(RUN_ROOT / "results" / "pareto_information_scaling.png")],
    ]
    for command in commands:
        outcome = subprocess.run(command, cwd=RUN_ROOT, text=True,
                                 stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        print(outcome.stdout[-5000:])
        if outcome.returncode:
            RUN_OK = False
            record("experiment", "failed", f"exit={outcome.returncode}")
            break
    if RUN_OK:
        summary = json.loads((RUN_ROOT / "results" / "pareto_information_summary.json").read_text())
        if not (summary["all_checked_configurations_meet_bound"] and
                summary["all_checked_configurations_meet_upper_bound"]):
            RUN_OK = False
            record("experiment", "failed", "numerical bound check failed")
        else:
            record("experiment", "passed", f"25 configurations; seconds={time.perf_counter()-started:.2f}")
else:
    record("experiment", "skipped", "earlier stage failed")

## 4. Zip every THM-06 output


In [ ]:
(RUN_ROOT / "notebook_stages.json").write_text(
    json.dumps({"run_ok": RUN_OK, "stages": STAGES}, indent=2), encoding="utf-8")
FINAL_ZIP = WORK_ROOT / "result_jcam_thm06_pareto_information.zip"
with zipfile.ZipFile(FINAL_ZIP, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(RUN_ROOT.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=(Path("thm06_pareto_information") / path.relative_to(RUN_ROOT)).as_posix())
print("Output archive:", FINAL_ZIP, "bytes:", FINAL_ZIP.stat().st_size, "run_ok:", RUN_OK)